# 05 — Requirement vs policy: high scalers, throughput and turnaround

**High scalers** (wall time ≤ 0.8× per doubling of cores) are where the clusters' parallel hardware can give
the most benefit. Here, *requirement vs reality* means the cores they can usefully use against the largest job the
queues allow, and the estimated wall time at each size.

**Low and intermediate scalers** are limited by turnaround, per-user concurrency and queue limits. Here, *requirement
vs reality* means the queue their typical job fits, the concurrency they want against the per-user limit, and how long a
year of their work would take at that limit.

*Principles: P4, P5, P6 in the [analysis framework](https://github.com/svaradh/hpc-questionnaire/blob/main/docs/analysis-framework.md). Nothing here scores or ranks groups; flags are candidates for the committee to look at. Totals cover responding groups only, and "Don't know" is never counted as zero; a yes/no flag shown as `<NA>` means its inputs are unknown. Only the respondents' answers and the cluster configuration are used — nothing depends on which code a workload runs.*

**To run:** paste the Spreadsheet ID in the first code cell, then *Runtime → Run all*. Colab asks once per session to sign in with your IISER Google account.

In [ ]:
SPREADSHEET_ID = 'PASTE_YOUR_SPREADSHEET_ID_HERE'   # <- the long string in the sheet URL
REPO_REF = 'main'                                   # git branch/tag for the helper code and cluster config

import os, urllib.request
if not os.path.exists('hpc_analysis.py'):            # in Colab: fetch the helper modules once per session
    for f in ['sheets_client.py', 'hpc_analysis.py', 'hpc_plots.py', 'code_capabilities.json']:
        urllib.request.urlretrieve(f'https://raw.githubusercontent.com/svaradh/hpc-questionnaire/{REPO_REF}/analysis/{f}', f)

import pandas as pd
from sheets_client import load_data, load_config
import hpc_analysis as ha
import hpc_plots as hp
pd.set_option('display.max_colwidth', 120)

dfs = load_data(SPREADSHEET_ID)
cfg = load_config(ref=REPO_REF)
wl = ha.workloads(dfs, cfg)
cov = ha.coverage(dfs)
print(f"{cov['groups']} groups responded; {cov['workloads']} workloads described "
      f"({cov['groups_with_workloads']} groups with at least one).")

## High scalers: cores wanted vs the largest job allowed

In [ ]:
hs = ha.high_scalers(wl, cfg)
if len(hs):
    hp.paired(hs.assign(label=hs['pi_name'] + ' · ' + hs['entry']), 'label', 'wanted_cores', 'policy_max_cores',
              'Cores usefully used', 'Largest job allowed', 'High scalers: useful job size vs policy', 'CPU cores')
hs.round(1)

`walltime_at_policy_max_h` estimates the wall time if the job is capped at the largest allowed size (from the measured or
reported scaling); compare it with `multi_node_limit_h` (72 h for multi-node jobs). `campaign_over_ceiling` marks
coordinated campaigns needing more cores at once than one user can hold in any single queue.

## Low and intermediate scalers: queue fit, concurrency and time to finish a year's work

In [ ]:
tp = ha.throughput(wl)
hp.counts(tp['queue'], 'Typical job: the queue it fits ("none fits" = no queue allows this shape; "unknown" = size or wall time not given)')
tp.round(2)

`years_to_finish` = (jobs per year × wall time) / (per-user job limit of that queue × 8,760 h): above ~0.5, the
per-user limit, not the hardware, paces the work. `concurrency_gap` = jobs wanted at once − jobs allowed at once.

In [ ]:
thr = tp.dropna(subset=['years_to_finish']).sort_values('years_to_finish', ascending=False)
if len(thr):
    hp.barh(thr['pi_name'] + ' · ' + thr['entry'], thr['years_to_finish'].round(2), 'Years to finish one year of work at the per-user job limit', 'Years')

## Queue waits

Each entry records the queue and the resources requested. *Expansion factor* = (typical wait + requested wall time) /
requested wall time; requested wall time stands in for run time, so this slightly understates it.

In [ ]:
qw = ha.queue_waits(dfs, wl, cfg)
qw.round(2)

## Debug and interactive use vs the 30-minute debug queues

In [ ]:
ha.debug_demand(dfs, cfg).round(2)